# 02 Clean

Turns the raw Vektis and CBS files into clean tables. Every cleaning decision is written down: what I found, what I did, why.

In [1]:
# pandas: the library for working with tables
import pandas as pd

In [2]:
# Load the raw Vektis file (semicolon-separated). The raw file itself is never changed.
vektis = pd.read_csv("../data/raw/Vektis Open Databestand Zorgverzekeringswet 2024 - gemeente.csv", sep=";")

In [3]:
# First look at the top rows, including the unlabelled bucket row (row 0)
vektis.head()

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,NaN,NaN,NaN,412394,220748.91,75738481.61,11107600.71,6081467.62,2120101.39,1852201.82,...,233868.73,276735.8,1297283.78,53388.62,760262.82,0.0,489674.26,9488679.78,2134088.05,641623.15
1,M,0 t/m 4 jaar,AA EN HUNZE,503,462.82,594197.18,32444.90,35500.43,20536.52,15357.50,...,0.00,0.0,2453.22,0.00,0.00,0.0,23236.19,0.00,0.00,0.00
2,M,0 t/m 4 jaar,AALSMEER,731,665.75,1829500.58,55985.58,51112.38,25609.76,17045.38,...,0.00,0.0,24028.45,0.00,0.00,0.0,125837.16,0.00,0.00,0.00
3,M,0 t/m 4 jaar,AALTEN,662,596.80,862338.30,45558.63,45592.74,27044.28,7194.71,...,0.00,0.0,1549.40,0.00,0.00,0.0,123649.13,0.00,0.00,0.00
4,M,0 t/m 4 jaar,ACHTKARSPELEN,680,619.29,903357.49,41051.42,46985.28,21486.90,11042.53,...,0.00,0.0,1147.02,0.00,0.00,0.0,46237.53,0.00,0.00,0.00


## Step 1. Privacy bucket

In [4]:
# Privacy bucket: groups under 10 people (hidden so no one can be identified)
# plus people with unknown sex, age or municipality. All lumped into one unlabelled row.
# Find it: the only row with no municipality name
bucket = vektis[vektis['gemeentenaam'].isna()]
bucket

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,NaN,NaN,NaN,412394,220748.91,75738481.61,11107600.71,6081467.62,2120101.39,1852201.82,...,233868.73,276735.8,1297283.78,53388.62,760262.82,0.0,489674.26,9488679.78,2134088.05,641623.15


In [5]:
# Size in people: the bucket's share of everyone insured
bucket_people = bucket["aantal_bsn"].sum()
total_people = vektis["aantal_bsn"].sum()
share = bucket_people / total_people * 100
print(f"Bucket: {bucket_people:,.0f} people, {share:.1f}% of all people")

Bucket: 412,394 people, 2.2% of all people


In [6]:
# All 26 cost columns, and their grand total (bucket still included at this point)
costs = vektis.filter(like="kosten")
total_costs = costs.sum().sum()

In [7]:
# Size in euros: the bucket's share of all cost
bucket_cost = bucket.filter(like="kosten").sum().sum()
cost_share = bucket_cost / total_costs * 100
print(f"Bucket: €{bucket_cost:,.0f}, {cost_share:.1f}% of all cost")

Bucket: €167,835,544, 0.3% of all cost


**Decision: privacy bucket**
- Found: one row with empty sex, age and municipality. Vektis puts groups under 10 people (privacy) and people with unknown sex, age or municipality here. It holds 412,394 people (2.2%) but only €167.8 million (0.3% of all cost).
- Did: removed the row. The raw file stays unchanged.
- Why: every KPI needs a municipality, and expected cost needs age and sex. This row has none of them, so it cannot be placed anywhere. Rejected: spreading it over municipalities, because we do not know where these people live.
- Effect: only 5 small cells are missing (Rozendaal, Schiermonnikoog, Vlieland). The bucket is cheap per person (about €760 per insured-year against €3,382), likely because many were insured for only part of the year. Removing it barely changes the national figures.

In [8]:
# Remove the bucket: keep only rows that have a municipality name
vektis = vektis[vektis["gemeentenaam"].notna()]

In [9]:
# Check the size: 12,991 rows expected
vektis.shape

(12991, 31)

In [10]:
# Stop with an error if the row count is not 12,991
assert len(vektis) == 12991

## Step 2. Negative costs

In [11]:
# Recreate after dropping the bucket, so costs and vektis have the same rows
costs = vektis.filter(like="kosten")
total_costs = costs.sum().sum()

In [12]:
# How many negative cells?
neg_costs = costs < 0
neg_costs.sum().sum()

np.int64(2)

In [13]:
# Which rows are they, and how big?
neg_rows = vektis[neg_costs.any(axis=1)]
neg_rows[["geslacht", "leeftijdsklasse", "gemeentenaam",
          "kosten_integrale_geboortezorg", "kosten_intramuraal_verblijf_ggz"]]

,geslacht,leeftijdsklasse,gemeentenaam,kosten_integrale_geboortezorg,kosten_intramuraal_verblijf_ggz
8485,V,25 t/m 29 jaar,UITGEEST,202928.60,-2110.60
8824,V,30 t/m 34 jaar,TWENTERAND,-2299.93,10266.52


In [14]:
# Total cost per row (26 columns added across), then per municipality.
# Compare the two negative cells with their municipality's total.
row_total = costs.sum(axis=1)
town_total = row_total.groupby(vektis["gemeentenaam"]).sum()
town_total[["UITGEEST", "TWENTERAND"]]

gemeentenaam
UITGEEST      4.323860e+07
TWENTERAND    1.273882e+08
dtype: float64

**Decision: negative costs**
- Found: 2 negative cells out of about 338,000: Uitgeest, women 25-29, inpatient mental health stays (−€2,111), and Twenterand, women 30-34, birth care (−€2,300). Each is under 0.01% of that municipality's total cost.
- Did: kept them as they are.
- Why: they are real corrections (reversed or refunded claims), too small to move any result, and keeping them means totals match the official Vektis figures. Rejected: setting them to zero (totals would no longer match Vektis) and dropping the rows (would also remove 25 real cost cells per row).

## Step 3. Care abroad

In [15]:
# Care abroad: total euros, and its share of all cost
abroad = vektis["kosten_grensoverschrijdende_zorg"]
abroad_costs = abroad.sum()
abroad_share = abroad_costs / total_costs * 100
print(f"Care abroad: €{abroad_costs:,.0f}, {abroad_share:.1f}% of all cost")

Care abroad: €347,307,725, 0.6% of all cost


In [16]:
# Care abroad as a share of each municipality's own cost, largest first
abroad_town = vektis["kosten_grensoverschrijdende_zorg"].groupby(vektis["gemeentenaam"]).sum()
abroad_pct = abroad_town / town_total * 100
abroad_pct.sort_values(ascending=False)

gemeentenaam
SLUIS              18.394525
HULST              10.182345
TERNEUZEN           8.246225
VAALS               4.936495
VEERE               2.590779
                     ...    
STADSKANAAL         0.143058
AALTEN              0.127767
NEDER-BETUWE        0.115078
PEKELA              0.101732
SCHIERMONNIKOOG     0.085287
Length: 342, dtype: float64

In [17]:
# The typical municipality, to compare the top ones against
abroad_pct.median()

np.float64(0.4037382248561877)

**Decision: care abroad**
- Found: care abroad (`kosten_grensoverschrijdende_zorg`) is €347.3 million, 0.6% of all cost, and the leaflet says it is incomplete because foreign bills arrive late. It is concentrated in border municipalities: median 0.4% of a municipality's cost, but Sluis 18%, Hulst 10%, Terneuzen 8% (Zeeuws-Vlaanderen, next to Belgium) and Vaals 5% (next to Germany and Belgium).
- Did: kept the column.
- Why: for border towns, care abroad is part of their normal hospital care. Excluding it would make Sluis look about 18% cheaper than it is and give the purchasing team a false signal. Rejected: excluding the column.
- Limitation: because the column is incomplete, border towns' true cost is probably higher than shown. This goes in the README's Limitations.

## Step 4. Age band as a number

In [18]:
# The problem: sorted as text, '5 t/m 9' lands after '45 t/m 49'
sorted(vektis["leeftijdsklasse"].unique())

['0 t/m  4 jaar',
 '10 t/m 14 jaar',
 '15 t/m 19 jaar',
 '20 t/m 24 jaar',
 '25 t/m 29 jaar',
 '30 t/m 34 jaar',
 '35 t/m 39 jaar',
 '40 t/m 44 jaar',
 '45 t/m 49 jaar',
 '5 t/m  9 jaar',
 '50 t/m 54 jaar',
 '55 t/m 59 jaar',
 '60 t/m 64 jaar',
 '65 t/m 69 jaar',
 '70 t/m 74 jaar',
 '75 t/m 79 jaar',
 '80 t/m 84 jaar',
 '85 t/m 89 jaar',
 '90+']

In [19]:
# New column: the first number in each label (the lower age), as a real number
vektis["age_from"] = vektis["leeftijdsklasse"].str.extract(r"(\d+)").astype(int)

In [20]:
# Check: each label once, ordered by the new number (expect 19 rows, 0 to 90)
vektis[["leeftijdsklasse", "age_from"]].drop_duplicates().sort_values("age_from")

,leeftijdsklasse,age_from
1,0 t/m 4 jaar,0
343,5 t/m 9 jaar,5
685,10 t/m 14 jaar,10
1027,15 t/m 19 jaar,15
1369,20 t/m 24 jaar,20
1711,25 t/m 29 jaar,25
2053,30 t/m 34 jaar,30
2395,35 t/m 39 jaar,35
2737,40 t/m 44 jaar,40
3079,45 t/m 49 jaar,45


**Note: age order.** Age bands are text, so they sort wrongly ("5 t/m 9" after "45 t/m 49"). Added `age_from`, the lower age as a number (0, 5 ... 90), for ordering and charts. The text label stays for display.

## Step 5. Care-type groups

In [21]:
# Load the mapping: which of the 10 groups each of the 26 cost columns belongs to
groups = pd.read_csv("../data/manual/care_type_groups.csv")
groups["group"].unique()

<StringArray>
[             'Hospital care',                   'Pharmacy',
                    'GP care',              'Mental health',
 'Nursing and rehabilitation',                'Paramedical',
            'Aids and dental',                  'Maternity',
                'Care abroad',        'Transport and other']
Length: 10, dtype: str

In [22]:
# For each of the 10 groups: find its cost columns and add them up per row
for group in groups["group"].unique():
    cols = groups.loc[groups["group"] == group, "column"]    # that group's column names, from the CSV
    vektis[group] = vektis[cols].sum(axis=1)                 # add that group's columns per row

In [23]:
# Check: the 10 new group columns, first rows
vektis[groups["group"].unique()].head()


,Hospital care,Pharmacy,GP care,Mental health,Nursing and rehabilitation,Paramedical,Aids and dental,Maternity,Care abroad,Transport and other
1,594197.18,32444.90,132131.44,0.0,2453.22,28642.36,27515.81,0.0,531.63,35783.67
2,1829500.58,55985.58,190685.70,0.0,24028.45,58198.07,54287.27,0.0,5443.40,147467.87
3,862338.30,45558.63,147475.81,0.0,1549.40,102185.03,31574.07,0.0,5020.35,138597.26
4,903357.49,41051.42,160024.85,0.0,1147.02,78934.60,69470.60,0.0,0.00,63371.88
5,1046813.52,59368.82,121608.84,0.0,109152.08,50877.47,55889.22,0.0,1969.44,30862.67


In [24]:
# Total of all 26 cost columns per row
vektis["cost_total"] = costs.sum(axis=1)

In [25]:
# Check the grouping: the 10 group columns must add up to the original total in every row.
# If a column was forgotten in the CSV, its cost would disappear;
# if a column was listed twice, its cost would be counted double.
# Allow less than 1 cent difference, because computers round decimals slightly.
group_names = groups["group"].unique()
difference = (vektis[group_names].sum(axis=1) - vektis["cost_total"]).abs().max()
print(f"Largest difference: €{difference:.6f}")
assert difference < 0.01


Largest difference: €0.000000


**Decision: care-type groups**
- Found: Vektis splits cost into 26 columns, too detailed to compare municipalities on. The leaflet (section 4, pages 7 to 10) defines what each column covers.
- Did: mapped the 26 columns to 10 care-type groups in `data/manual/care_type_groups.csv`, then added one column per group plus `cost_total`. Check: the 10 groups add up to the total of all 26 columns in every row (largest difference €0.000000).
- Why: 10 groups are few enough to read in one chart, and each is a kind of care the purchasing team actually buys. Two placements came from the leaflet: `mdz` (multidisciplinary care, e.g. diabetes programmes) goes in GP care, and `gzsp` goes in nursing and rehabilitation. Care abroad is its own group because it is 18% of Sluis's cost and would distort hospital care there.
- Rejected: keeping all 26 columns (unreadable) and typing the mapping in code (a CSV is easier to check and to change).


## Step 6. Name matching

In [26]:
# Each Vektis municipality name once, sorted A to Z (12,991 rows, but only 342 towns)
vektis_names = sorted(vektis["gemeentenaam"].unique())

In [27]:
# Check: how many different Vektis names? Expect 342
len(vektis_names)

342

In [28]:
# Load the CBS municipality list: code (Key) and name (Title), 342 rows
regions = pd.read_csv("../data/raw/cbs_86137NED_regions.csv")

In [29]:
# Check: how many CBS names? Expect 342.
len(regions)

342

In [30]:
# CBS names sorted A to Z
cbs_names = sorted(regions["Title"])

In [31]:
# Look at the CBS names, to compare with Vektis
cbs_names

["'s-Gravenhage (gemeente)",
 "'s-Hertogenbosch",
 'Aa en Hunze',
 'Aalsmeer',
 'Aalten',
 'Achtkarspelen',
 'Alblasserdam',
 'Albrandswaard',
 'Alkmaar',
 'Almelo',
 'Almere',
 'Alphen aan den Rijn',
 'Alphen-Chaam',
 'Altena',
 'Ameland',
 'Amersfoort',
 'Amstelveen',
 'Amsterdam',
 'Apeldoorn',
 'Arnhem',
 'Assen',
 'Asten',
 'Baarle-Nassau',
 'Baarn',
 'Barendrecht',
 'Barneveld',
 'Beek (L.)',
 'Beekdaelen',
 'Beesel',
 'Berg en Dal',
 'Bergeijk',
 'Bergen (L.)',
 'Bergen (NH.)',
 'Bergen op Zoom',
 'Berkelland',
 'Bernheze',
 'Best',
 'Beuningen',
 'Beverwijk',
 'Bladel',
 'Blaricum',
 'Bloemendaal',
 'Bodegraven-Reeuwijk',
 'Boekel',
 'Borger-Odoorn',
 'Borne',
 'Borsele',
 'Boxtel',
 'Breda',
 'Bronckhorst',
 'Brummen',
 'Brunssum',
 'Bunnik',
 'Bunschoten',
 'Buren',
 'Capelle aan den IJssel',
 'Castricum',
 'Coevorden',
 'Cranendonck',
 'Culemborg',
 'Dalfsen',
 'Dantumadiel',
 'De Bilt',
 'De Fryske Marren',
 'De Ronde Venen',
 'De Wolden',
 'Delft',
 'Den Helder',
 'Deurne'

In [32]:
# Look at the Vektis names, to compare with CBS
vektis_names

['AA EN HUNZE',
 'AALSMEER',
 'AALTEN',
 'ACHTKARSPELEN',
 'ALBLASSERDAM',
 'ALBRANDSWAARD',
 'ALKMAAR',
 'ALMELO',
 'ALMERE',
 'ALPHEN AAN DEN RIJN',
 'ALPHEN-CHAAM',
 'ALTENA',
 'AMELAND',
 'AMERSFOORT',
 'AMSTELVEEN',
 'AMSTERDAM',
 'APELDOORN',
 'ARNHEM',
 'ASSEN',
 'ASTEN',
 'BAARLE-NASSAU',
 'BAARN',
 'BARENDRECHT',
 'BARNEVELD',
 'BEEK',
 'BEEKDAELEN',
 'BEESEL',
 'BERG EN DAL',
 'BERGEIJK',
 'BERGEN LB',
 'BERGEN NH',
 'BERGEN OP ZOOM',
 'BERKELLAND',
 'BERNHEZE',
 'BEST',
 'BEUNINGEN',
 'BEVERWIJK',
 'BLADEL',
 'BLARICUM',
 'BLOEMENDAAL',
 'BODEGRAVEN-REEUWIJK',
 'BOEKEL',
 'BORGER-ODOORN',
 'BORNE',
 'BORSELE',
 'BOXTEL',
 'BREDA',
 'BRONCKHORST',
 'BRUMMEN',
 'BRUNSSUM',
 'BUNNIK',
 'BUNSCHOTEN',
 'BUREN',
 'CAPELLE AAN DEN IJSSEL',
 'CASTRICUM',
 'COEVORDEN',
 'CRANENDONCK',
 'CULEMBORG',
 'DALFSEN',
 'DANTUMADIEL',
 'DE BILT',
 'DE FRYSKE MARREN',
 'DE RONDE VENEN',
 'DE WOLDEN',
 'DELFT',
 'DEN HELDER',
 'DEURNE',
 'DEVENTER',
 'DIEMEN',
 'DIJK EN WAARD',
 'DINKELLAND',
 

In [33]:
# Matching key for town names: capitals, accents removed.
# One function for both sides, so CBS and Vektis follow exactly the same rule.
def make_match_key(names):
    return (
        names
        .str.upper()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
    )

In [34]:
# Give both tables a match_key column, made by the same function
vektis["match_key"] = make_match_key(vektis["gemeentenaam"])
regions["match_key"] = make_match_key(regions["Title"])

In [35]:
# Check: both sides should have 342 different match keys
print("Vektis:", len(vektis["match_key"].unique()))
print("CBS:", len(regions["match_key"].unique()))


Vektis: 342
CBS: 342


In [36]:
# Check: the accented Frisian names should now have plain letters in match_key
regions[regions["Title"].str.contains("Fryslân")][["Title", "match_key"]]

,Title,match_key
196,Noardeast-Fryslân,NOARDEAST-FRYSLAN
266,Súdwest-Fryslân,SUDWEST-FRYSLAN


In [38]:
# Small tables with one row per town, without duplicates
vektis_towns = vektis[["match_key", "gemeentenaam"]].drop_duplicates()
cbs_towns = regions[["match_key", "Title", "Key"]]

# Join on match_key. how="left" keeps every Vektis town, even without a CBS partner;
# indicator=True adds a _merge column saying where each row was found
towns = pd.merge(vektis_towns, cbs_towns, on="match_key", how="left", indicator=True)

# Check: how many matched (both) and how many did not (left_only)
towns["_merge"].value_counts()

_merge
both          329
left_only      13
right_only      0
Name: count, dtype: int64

In [40]:
# The Vektis towns that found no CBS partner: these need a manual fix
towns[towns["_merge"] == "left_only"]


,match_key,gemeentenaam,Title,Key,_merge
24,BEEK,BEEK,NaN,NaN,left_only
29,BERGEN LB,BERGEN LB,NaN,NaN,left_only
30,BERGEN NH,BERGEN NH,NaN,NaN,left_only
106,GRONINGEN,GRONINGEN,NaN,NaN,left_only
127,HENGELO,HENGELO,NaN,NaN,left_only
157,LAREN,LAREN,NaN,NaN,left_only
180,MIDDELBURG,MIDDELBURG,NaN,NaN,left_only
201,NUENEN CA,NUENEN CA,NaN,NaN,left_only
237,RIJSWIJK,RIJSWIJK,NaN,NaN,left_only
244,S GRAVENHAGE,S GRAVENHAGE,NaN,NaN,left_only


In [47]:
# Search CBS for the Vektis name BEEK, to find its code
cbs_towns[cbs_towns["match_key"].str.contains("BEEK")]

,match_key,Title,Key
24,BEEK (L.),Beek (L.),GM0888
25,BEEKDAELEN,Beekdaelen,GM1954
131,HILVARENBEEK,Hilvarenbeek,GM0798
152,LAARBEEK,Laarbeek,GM1659
